# Predicting Heart Disease - S6E2
LightGBM + XGBoost + CatBoost → Rank Averaging Ensemble

**v2: 原データ追加版**

In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

In [6]:
class CFG:
    seed = 42
    n_splits = 5
    target_col = 'Heart Disease'
    use_original = True  # ← 原データ追加のON/OFF

class Paths:
    p = '/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data'
    train = p + '/train.csv'
    test = p + '/test.csv'
    sample = p + '/sample_submission.csv'
    # 原データ（Kaggleからダウンロードしたファイル名に合わせて変更してください）
    original = p + '/dataset_heart.csv'

## 1. Data Loading + 原データ追加

In [7]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

print(f'Train shape (synthetic only): {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Train columns: {train.columns.tolist()}')

Train shape (synthetic only): (630000, 15)
Test shape: (270000, 14)
Train columns: ['id', 'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease']


In [8]:
# print(original.columns.tolist())
# print(original.head())

In [9]:
if CFG.use_original:
    original = pd.read_csv(Paths.original)
    print(f'Original shape: {original.shape}')
    print(f'Original columns: {original.columns.tolist()}')

    # カラム名を完全にリネーム（原データ → コンペデータ）
    rename_map = {
        'age': 'Age',
        'sex ': 'Sex',                                    # 末尾にスペースあり注意
        'chest pain type': 'Chest pain type',
        'resting blood pressure': 'BP',
        'serum cholestoral': 'Cholesterol',
        'fasting blood sugar': 'FBS over 120',
        'resting electrocardiographic results': 'EKG results',
        'max heart rate': 'Max HR',
        'exercise induced angina': 'Exercise angina',
        'oldpeak': 'ST depression',
        'ST segment': 'Slope of ST',
        'major vessels': 'Number of vessels fluro',
        'thal': 'Thallium',
        'heart disease': 'Heart Disease',
    }
    original = original.rename(columns=rename_map)

    # ターゲット変換（Statlog形式: 1=Absence→0, 2=Presence→1）
    original['Heart Disease'] = original['Heart Disease'].map({1: 0, 2: 1})

    print(f'\nAfter rename columns: {original.columns.tolist()}')
    print(f'Target unique: {original["Heart Disease"].unique()}')
    print(f'Target distribution:\n{original["Heart Disease"].value_counts()}')
    print(f'NaN in target: {original["Heart Disease"].isnull().sum()}')

Original shape: (270, 14)
Original columns: ['age', 'sex ', 'chest pain type', 'resting blood pressure', 'serum cholestoral', 'fasting blood sugar', 'resting electrocardiographic results', 'max heart rate', 'exercise induced angina', 'oldpeak', 'ST segment', 'major vessels', 'thal', 'heart disease']

After rename columns: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease']
Target unique: [1 0]
Target distribution:
Heart Disease
0    150
1    120
Name: count, dtype: int64
NaN in target: 0


In [10]:
# 原データをtrainに結合
if CFG.use_original:
    # 原データにidカラムがない場合は追加（trainのid最大値+1から連番）
    if 'id' not in original.columns:
        max_id = train['id'].max()
        original['id'] = range(max_id + 1, max_id + 1 + len(original))

    # カラム順序をtrainに合わせる
    original = original[train.columns]

    # 結合
    print(f'Before concat: train={len(train)}')
    train = pd.concat([train, original], axis=0, ignore_index=True)
    print(f'After concat:  train={len(train)} (+{len(original)} original rows)')

    # 完全重複行の除去（原データの一部がsynthetic dataに含まれている場合がある）
    cols_for_dedup = [c for c in train.columns if c != 'id']
    n_before = len(train)
    train = train.drop_duplicates(subset=cols_for_dedup, keep='first').reset_index(drop=True)
    n_after = len(train)
    print(f'Dedup: {n_before} → {n_after} (removed {n_before - n_after} duplicates)')

Before concat: train=630000
After concat:  train=630270 (+270 original rows)
Dedup: 630270 → 630270 (removed 0 duplicates)


## 2. Target Encoding

In [11]:
# ターゲットを0/1に変換
if train[CFG.target_col].dtype == 'object' or train[CFG.target_col].dtype.name == 'category':
    target_map = {
        'Absence': 0, 'Presence': 1,
        'No': 0, 'Yes': 1,
        0: 0, 1: 1,
    }
    train[CFG.target_col] = train[CFG.target_col].map(target_map)

# 変換確認
print(f'Target dtype: {train[CFG.target_col].dtype}')
print(f'Target unique: {train[CFG.target_col].unique()}')
print(f'Target distribution:\n{train[CFG.target_col].value_counts(normalize=True)}')
assert train[CFG.target_col].isnull().sum() == 0, 'ERROR: NaN in target!'

Target dtype: int64
Target unique: [1 0]
Target distribution:
Heart Disease
0    0.551662
1    0.448338
Name: proportion, dtype: float64


## 3. Feature Engineering

In [12]:
def add_features(df):
    """train / test に同一の特徴量変換を適用する関数"""

    # 型変換: int8/int16 → int32 でオーバーフロー防止
    for col in df.select_dtypes(include=['int8', 'int16']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float16']).columns:
        df[col] = df[col].astype('float32')

    # --- 数値 x 数値 ---
    df['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    df['Age_div_MaxHR'] = df['Age'] / (df['Max HR'] + 1e-5)
    df['STdep_x_MaxHR'] = df['ST depression'] * df['Max HR']
    df['BP_x_Chol'] = df['BP'] * df['Cholesterol']
    df['Age_x_STdep'] = df['Age'] * df['ST depression']
    df['Age_x_Vessels'] = df['Age'] * df['Number of vessels fluro']
    df['BP_x_MaxHR'] = df['BP'] * df['Max HR']
    df['Chol_x_Age'] = df['Cholesterol'] * df['Age']
    df['STdep_x_Vessels'] = df['ST depression'] * df['Number of vessels fluro']

    # --- 数値 x カテゴリ ---
    df['Sex_x_MaxHR'] = df['Sex'] * df['Max HR']
    df['Sex_x_Chol'] = df['Sex'] * df['Cholesterol']
    df['ChestPain_x_STdep'] = df['Chest pain type'] * df['ST depression']
    df['ChestPain_x_MaxHR'] = df['Chest pain type'] * df['Max HR']
    df['ExAngina_x_STdep'] = df['Exercise angina'] * df['ST depression']
    df['ExAngina_x_MaxHR'] = df['Exercise angina'] * df['Max HR']
    df['Thallium_x_Vessels'] = df['Thallium'] * df['Number of vessels fluro']
    df['SlopeSTx_STdep'] = df['Slope of ST'] * df['ST depression']

    # --- 差分・比率 ---
    df['MaxHR_reserve'] = (220 - df['Age']) - df['Max HR']
    df['MaxHR_ratio'] = df['Max HR'] / (220 - df['Age'] + 1e-5)

    # --- カテゴリ x カテゴリ ---
    df['ChestPain_ExAngina'] = df['Chest pain type'].astype(str) + '_' + df['Exercise angina'].astype(str)
    df['ChestPain_Thallium'] = df['Chest pain type'].astype(str) + '_' + df['Thallium'].astype(str)
    df['SlopeST_ExAngina'] = df['Slope of ST'].astype(str) + '_' + df['Exercise angina'].astype(str)

    return df

train = add_features(train)
test = add_features(test)
print(f'Train shape after FE: {train.shape}')
print(f'Test shape after FE: {test.shape}')

Train shape after FE: (630270, 37)
Test shape after FE: (270000, 36)


## 4. Column Definitions

In [13]:
drop_cols = ['id', CFG.target_col]
feature_cols = [c for c in train.columns if c not in drop_cols]
cat_cols = train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

y_train = train[CFG.target_col]
print(f'\ny_train dtype: {y_train.dtype}')
print(f'y_train unique: {y_train.unique()}')
print(f'y_train shape: {y_train.shape}')

Feature columns (35): ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Age_x_MaxHR', 'Age_div_MaxHR', 'STdep_x_MaxHR', 'BP_x_Chol', 'Age_x_STdep', 'Age_x_Vessels', 'BP_x_MaxHR', 'Chol_x_Age', 'STdep_x_Vessels', 'Sex_x_MaxHR', 'Sex_x_Chol', 'ChestPain_x_STdep', 'ChestPain_x_MaxHR', 'ExAngina_x_STdep', 'ExAngina_x_MaxHR', 'Thallium_x_Vessels', 'SlopeSTx_STdep', 'MaxHR_reserve', 'MaxHR_ratio', 'ChestPain_ExAngina', 'ChestPain_Thallium', 'SlopeST_ExAngina']
Categorical columns (3): ['ChestPain_ExAngina', 'ChestPain_Thallium', 'SlopeST_ExAngina']

y_train dtype: int64
y_train unique: [1 0]
y_train shape: (630270,)


## 5. CV Split

In [14]:
cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
fold_indices = list(cv.split(train[feature_cols], y_train))
print(f'CV: {CFG.n_splits}-fold StratifiedKFold')
for i, (tr_idx, va_idx) in enumerate(fold_indices):
    print(f'  Fold {i+1}: train={len(tr_idx)}, val={len(va_idx)}')

CV: 5-fold StratifiedKFold
  Fold 1: train=504216, val=126054
  Fold 2: train=504216, val=126054
  Fold 3: train=504216, val=126054
  Fold 4: train=504216, val=126054
  Fold 5: train=504216, val=126054


## 6. MODEL 1: LightGBM

In [15]:
x_train_lgb = train[feature_cols].copy()
x_test_lgb = test[feature_cols].copy()
for col in cat_cols:
    x_train_lgb[col] = x_train_lgb[col].astype('category')
    x_test_lgb[col] = x_test_lgb[col].astype('category')

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'n_estimators': 10000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': CFG.seed,
    'n_jobs': -1,
}

oof_lgb = np.zeros(len(x_train_lgb))
test_lgb = np.zeros(len(x_test_lgb))

for nfold, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'\n--- LightGBM Fold {nfold + 1} ---')
    x_tr = x_train_lgb.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    x_va = x_train_lgb.iloc[val_idx]
    y_va = y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(200),
        ],
    )

    oof_lgb[val_idx] = model.predict_proba(x_va)[:, 1]
    test_lgb += model.predict_proba(x_test_lgb)[:, 1] / CFG.n_splits
    print(f'  Fold {nfold + 1} AUC: {roc_auc_score(y_va, oof_lgb[val_idx]):.6f}')

oof_auc_lgb = roc_auc_score(y_train, oof_lgb)
print(f'\n★ LightGBM OOF AUC: {oof_auc_lgb:.6f}')


--- LightGBM Fold 1 ---
[200]	valid_0's auc: 0.955008
[400]	valid_0's auc: 0.955326
[600]	valid_0's auc: 0.955336
  Fold 1 AUC: 0.955354

--- LightGBM Fold 2 ---
[200]	valid_0's auc: 0.955104
[400]	valid_0's auc: 0.955305
[600]	valid_0's auc: 0.955318
  Fold 2 AUC: 0.955331

--- LightGBM Fold 3 ---
[200]	valid_0's auc: 0.953788
[400]	valid_0's auc: 0.954062
  Fold 3 AUC: 0.954078

--- LightGBM Fold 4 ---
[200]	valid_0's auc: 0.954712
[400]	valid_0's auc: 0.954939
[600]	valid_0's auc: 0.954993
  Fold 4 AUC: 0.955001

--- LightGBM Fold 5 ---
[200]	valid_0's auc: 0.955023
[400]	valid_0's auc: 0.955329
[600]	valid_0's auc: 0.955369
  Fold 5 AUC: 0.955371

★ LightGBM OOF AUC: 0.955026


## 7. MODEL 2: XGBoost

In [16]:
x_train_xgb = train[feature_cols].copy()
x_test_xgb = test[feature_cols].copy()

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([x_train_xgb[col].astype(str), x_test_xgb[col].astype(str)]))
    x_train_xgb[col] = le.transform(x_train_xgb[col].astype(str))
    x_test_xgb[col] = le.transform(x_test_xgb[col].astype(str))
    label_encoders[col] = le

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 10000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': CFG.seed,
    'n_jobs': -1,
    'tree_method': 'hist',
    'verbosity': 0,
    'early_stopping_rounds': 100,
}

oof_xgb = np.zeros(len(x_train_xgb))
test_xgb = np.zeros(len(x_test_xgb))

for nfold, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'\n--- XGBoost Fold {nfold + 1} ---')
    x_tr = x_train_xgb.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    x_va = x_train_xgb.iloc[val_idx]
    y_va = y_train.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        verbose=200,
    )

    oof_xgb[val_idx] = model.predict_proba(x_va)[:, 1]
    test_xgb += model.predict_proba(x_test_xgb)[:, 1] / CFG.n_splits
    print(f'  Fold {nfold + 1} AUC: {roc_auc_score(y_va, oof_xgb[val_idx]):.6f}')

oof_auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'\n★ XGBoost OOF AUC: {oof_auc_xgb:.6f}')


--- XGBoost Fold 1 ---
[0]	validation_0-auc:0.93897
[200]	validation_0-auc:0.95489
[400]	validation_0-auc:0.95533
[600]	validation_0-auc:0.95534
[629]	validation_0-auc:0.95534
  Fold 1 AUC: 0.955349

--- XGBoost Fold 2 ---
[0]	validation_0-auc:0.93913
[200]	validation_0-auc:0.95492
[400]	validation_0-auc:0.95536
[600]	validation_0-auc:0.95535
[611]	validation_0-auc:0.95535
  Fold 2 AUC: 0.955378

--- XGBoost Fold 3 ---
[0]	validation_0-auc:0.93551
[200]	validation_0-auc:0.95366
[400]	validation_0-auc:0.95403
[583]	validation_0-auc:0.95402
  Fold 3 AUC: 0.954048

--- XGBoost Fold 4 ---
[0]	validation_0-auc:0.93771
[200]	validation_0-auc:0.95449
[400]	validation_0-auc:0.95499
[523]	validation_0-auc:0.95501
  Fold 4 AUC: 0.955013

--- XGBoost Fold 5 ---
[0]	validation_0-auc:0.93819
[200]	validation_0-auc:0.95486
[400]	validation_0-auc:0.95534
[540]	validation_0-auc:0.95535
  Fold 5 AUC: 0.955363

★ XGBoost OOF AUC: 0.955028


## 8. MODEL 3: CatBoost

In [17]:
x_train_cat = train[feature_cols].copy()
x_test_cat = test[feature_cols].copy()

cat_col_indices = [feature_cols.index(col) for col in cat_cols]

cat_params = {
    'iterations': 10000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'eval_metric': 'AUC',
    'random_seed': CFG.seed,
    'verbose': 200,
    'early_stopping_rounds': 100,
    'task_type': 'CPU',
}

oof_cat = np.zeros(len(x_train_cat))
test_cat = np.zeros(len(x_test_cat))

for nfold, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'\n--- CatBoost Fold {nfold + 1} ---')
    x_tr = x_train_cat.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    x_va = x_train_cat.iloc[val_idx]
    y_va = y_train.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(
        x_tr, y_tr,
        eval_set=(x_va, y_va),
        cat_features=cat_col_indices,
    )

    oof_cat[val_idx] = model.predict_proba(x_va)[:, 1]
    test_cat += model.predict_proba(x_test_cat)[:, 1] / CFG.n_splits
    print(f'  Fold {nfold + 1} AUC: {roc_auc_score(y_va, oof_cat[val_idx]):.6f}')

oof_auc_cat = roc_auc_score(y_train, oof_cat)
print(f'\n★ CatBoost OOF AUC: {oof_auc_cat:.6f}')


--- CatBoost Fold 1 ---
0:	test: 0.9388820	best: 0.9388820 (0)	total: 143ms	remaining: 23m 46s
200:	test: 0.9545846	best: 0.9545846 (200)	total: 13.5s	remaining: 10m 58s
400:	test: 0.9551703	best: 0.9551703 (400)	total: 27.6s	remaining: 11m
600:	test: 0.9554139	best: 0.9554139 (600)	total: 42.5s	remaining: 11m 5s
800:	test: 0.9555065	best: 0.9555065 (800)	total: 58.3s	remaining: 11m 9s
1000:	test: 0.9555445	best: 0.9555462 (998)	total: 1m 13s	remaining: 10m 58s
1200:	test: 0.9555658	best: 0.9555667 (1199)	total: 1m 28s	remaining: 10m 48s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9555703161
bestIteration = 1274

Shrink model to first 1275 iterations.
  Fold 1 AUC: 0.955570

--- CatBoost Fold 2 ---
0:	test: 0.9391040	best: 0.9391040 (0)	total: 81.5ms	remaining: 13m 35s
200:	test: 0.9547023	best: 0.9547023 (200)	total: 14.6s	remaining: 11m 52s
400:	test: 0.9552807	best: 0.9552807 (400)	total: 29.4s	remaining: 11m 43s
600:	test: 0.9555240	best: 0.9555240 (600)	t

## 9. Ensemble: Rank Averaging

In [18]:
def rank_average(preds_list, weights=None):
    if weights is None:
        weights = [1.0 / len(preds_list)] * len(preds_list)
    ranked = [rankdata(p) / len(p) for p in preds_list]
    result = np.zeros_like(preds_list[0])
    for r, w in zip(ranked, weights):
        result += r * w
    return result

print('=== OOF AUC per model ===')
print(f'  LightGBM : {oof_auc_lgb:.6f}')
print(f'  XGBoost  : {oof_auc_xgb:.6f}')
print(f'  CatBoost : {oof_auc_cat:.6f}')

oof_equal = rank_average([oof_lgb, oof_xgb, oof_cat])
auc_equal = roc_auc_score(y_train, oof_equal)
print(f'\n  Ensemble (equal) OOF AUC: {auc_equal:.6f}')

print('\n--- Weight Optimization ---')
best_auc = 0
best_weights = None

for w1 in np.arange(0.1, 0.9, 0.05):
    for w2 in np.arange(0.1, 0.9 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        weights = [w1, w2, w3]
        oof_blend = rank_average([oof_lgb, oof_xgb, oof_cat], weights)
        auc = roc_auc_score(y_train, oof_blend)
        if auc > best_auc:
            best_auc = auc
            best_weights = weights

print(f'  Best weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}')
print(f'  Best OOF AUC: {best_auc:.6f}')

test_ensemble = rank_average([test_lgb, test_xgb, test_cat], best_weights)

=== OOF AUC per model ===
  LightGBM : 0.955026
  XGBoost  : 0.955028
  CatBoost : 0.955288

  Ensemble (equal) OOF AUC: 0.955241

--- Weight Optimization ---
  Best weights: LGB=0.10, XGB=0.10, CAT=0.80
  Best OOF AUC: 0.955302


## 10. Submission

In [19]:
submission = sample_submission.copy()
submission[CFG.target_col] = test_ensemble.astype(float)

print(f'Submission shape: {submission.shape}')
print(f'Submission dtypes:\n{submission.dtypes}')
print(f'Prediction range: [{submission[CFG.target_col].min():.6f}, {submission[CFG.target_col].max():.6f}]')
print(submission.head())

submission.to_csv('submission_ensemble_v2.csv', index=False)
print('\n✅ submission_ensemble_v2.csv saved!')

Submission shape: (270000, 2)
Submission dtypes:
id                 int64
Heart Disease    float64
dtype: object
Prediction range: [0.000007, 0.999981]
       id  Heart Disease
0  630000       0.794637
1  630001       0.065089
2  630002       0.890771
3  630003       0.024480
4  630004       0.447936

✅ submission_ensemble_v2.csv saved!


In [20]:
print('=' * 50)
print('FINAL SUMMARY')
print('=' * 50)
print(f'  Original data: {"ON" if CFG.use_original else "OFF"}')
print(f'  LightGBM OOF AUC : {oof_auc_lgb:.6f}')
print(f'  XGBoost  OOF AUC : {oof_auc_xgb:.6f}')
print(f'  CatBoost OOF AUC : {oof_auc_cat:.6f}')
print(f'  Ensemble OOF AUC : {best_auc:.6f}')
print(f'  Best weights      : LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}')
print('=' * 50)
print('\n→ submission_ensemble_v2.csv を提出してください')

FINAL SUMMARY
  Original data: ON
  LightGBM OOF AUC : 0.955026
  XGBoost  OOF AUC : 0.955028
  CatBoost OOF AUC : 0.955288
  Ensemble OOF AUC : 0.955302
  Best weights      : LGB=0.10, XGB=0.10, CAT=0.80

→ submission_ensemble_v2.csv を提出してください


In [21]:
# =============================================================================
# 提出パターン比較用
# =============================================================================

# パターン1: CatBoost単体（OOFで最強なので）
sub1 = sample_submission.copy()
sub1[CFG.target_col] = test_cat.astype(float)
sub1.to_csv('submission_catboost_only.csv', index=False)
print('✅ submission_catboost_only.csv saved!')

# パターン2: 等重み（過学習した重みよりLBで強い場合がある）
test_equal = rank_average([test_lgb, test_xgb, test_cat], [1/3, 1/3, 1/3])
sub2 = sample_submission.copy()
sub2[CFG.target_col] = test_equal.astype(float)
sub2.to_csv('submission_equal_weight.csv', index=False)
print('✅ submission_equal_weight.csv saved!')

# パターン3: CatBoost重め but 少し混ぜる
test_cat_heavy = rank_average([test_lgb, test_xgb, test_cat], [0.15, 0.15, 0.70])
sub3 = sample_submission.copy()
sub3[CFG.target_col] = test_cat_heavy.astype(float)
sub3.to_csv('submission_cat_heavy.csv', index=False)
print('✅ submission_cat_heavy.csv saved!')

✅ submission_catboost_only.csv saved!
✅ submission_equal_weight.csv saved!
✅ submission_cat_heavy.csv saved!
